# 00 — Attach OpenAlex work id to `Data/work_*.parquet`

**Why.** `OpenAlex/Works_oa.py` extracted the works corpus but its `READ_COLS`
omitted `id`, so `Data/work_*.parquet` (and every non-PPP paper sampled from it)
has **no OpenAlex work id** — which is why matched non-PPP control papers got
placeholder ids (`NONPPP_IDX_*`) and could not be joined to `sciscinet_papers.parquet`
(keyed by `paperid` = `W...`).

**Fix.** The raw dump `/project/jevans/tip/data/openalex/works.parquet` *does*
carry `id` (`https://openalex.org/W...`) alongside `title`, `publication_year`,
and `abstract_inverted_index`. This notebook recovers the id by matching on
**normalized title + publication_year** (same en, year≥1970 filter the corpus was
built with) and writes id-enriched copies to `Data/work_by_year_with_id/`.

Output id (`W...`) matches the sciscinet `paperid` format, so non-PPP papers can
then be joined to sciscinet metrics (enabling B.2c paper-side comparison).

**Compute:** CPU, Midway (one streaming pass over ~405 raw shards; run on a
big-memory node). Steps are resumable (skip if outputs exist).

In [1]:
import os, re, glob, time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

RAW_DUMP = "/project/jevans/tip/data/openalex/works.parquet"   # has id + title + abstract_inverted_index
WORK_DIR = "/project/jevans/Dawoon/OpenAlex/Data"              # work_{year}.parquet (no id) live here
SCI_DIR  = "/project/jevans/Dawoon/SciTech_PPP/Data"           # sciscinet_papers.parquet
MAP_PATH = os.path.join(WORK_DIR, "openalex_id_map.parquet")   # built below: (title_norm, year, openalex_id)
OUT_DIR  = os.path.join(WORK_DIR, "work_by_year_with_id")      # id-enriched outputs
os.makedirs(OUT_DIR, exist_ok=True)

MIN_YEAR = 1970
BATCH = 250_000

def norm_title(s: pd.Series) -> pd.Series:
    """Vectorized title normalization (lowercase, alnum-only, collapse spaces)."""
    return (s.fillna("").astype(str).str.lower()
             .str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip())

print("RAW shards:", len(glob.glob(os.path.join(RAW_DUMP, "*.parquet"))))

/home/jdwoon0523/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


RAW shards: 405


In [2]:
# ── Step 1: build the (title_norm, year) -> openalex_id map (streaming, low memory) ──
if os.path.exists(MAP_PATH):
    print(f"Map already exists: {MAP_PATH} (delete to rebuild)")
else:
    t0 = time.time()
    scanner = ds.dataset(RAW_DUMP).scanner(
        columns=["id", "title", "publication_year", "language"], batch_size=BATCH)
    writer = None; n_in = 0; n_out = 0
    for i, batch in enumerate(scanner.to_batches()):
        df = batch.to_pandas(); n_in += len(df)
        yr = pd.to_numeric(df["publication_year"], errors="coerce")
        df = df[(df["language"] == "en") & (yr >= MIN_YEAR)]
        if df.empty:
            continue
        out = pd.DataFrame({
            "title_norm": norm_title(df["title"]),
            "year": pd.to_numeric(df["publication_year"], errors="coerce").astype("Int64"),
            "openalex_id": df["id"].astype(str).str.rstrip("/").str.rsplit("/", n=1).str[-1],
        })
        out = out[(out["title_norm"] != "") & out["year"].notna()]
        if out.empty:
            continue
        tbl = pa.Table.from_pandas(out, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(MAP_PATH, tbl.schema)
        writer.write_table(tbl); n_out += len(out)
        if (i + 1) % 50 == 0:
            print(f"  batch {i+1}: scanned {n_in:,} -> kept {n_out:,}  ({time.time()-t0:.0f}s)")
    if writer: writer.close()
    print(f"Done. scanned {n_in:,}, wrote {n_out:,} id rows -> {MAP_PATH} in {time.time()-t0:.0f}s")

  batch 50: scanned 3,543,750 -> kept 2,398,470  (23s)
  batch 100: scanned 5,540,052 -> kept 4,152,932  (38s)
  batch 150: scanned 7,374,260 -> kept 5,772,101  (53s)
  batch 200: scanned 10,598,482 -> kept 8,198,313  (83s)
  batch 250: scanned 12,643,409 -> kept 9,954,710  (99s)
  batch 300: scanned 14,316,288 -> kept 11,515,293  (113s)
  batch 350: scanned 16,641,619 -> kept 13,394,872  (129s)
  batch 400: scanned 18,660,655 -> kept 15,182,508  (143s)
  batch 450: scanned 21,234,087 -> kept 17,172,840  (161s)
  batch 500: scanned 23,457,842 -> kept 19,042,021  (177s)
  batch 550: scanned 25,344,775 -> kept 20,746,453  (193s)
  batch 600: scanned 27,686,513 -> kept 22,671,190  (209s)
  batch 650: scanned 31,042,103 -> kept 24,900,108  (230s)
  batch 700: scanned 33,556,310 -> kept 26,951,223  (248s)
  batch 750: scanned 35,515,663 -> kept 28,721,635  (264s)
  batch 800: scanned 37,508,649 -> kept 30,491,196  (279s)
  batch 850: scanned 40,082,824 -> kept 32,339,293  (297s)
  batch 900

In [3]:
# ── Step 2: attach openalex_id to each work_{year}.parquet (per-year; ambiguous titles dropped) ──
map_ds = ds.dataset(MAP_PATH)
rows = []
for year in range(MIN_YEAR, 2026):
    wf = os.path.join(WORK_DIR, f"work_{year}.parquet")
    if not os.path.exists(wf):
        continue
    w = pd.read_parquet(wf)
    w["title_norm"] = norm_title(w["title"])
    # this year's id map; drop title_norm that map to >1 id (ambiguous) to avoid mis-links
    mp = map_ds.to_table(filter=ds.field("year") == year).to_pandas()
    mp = mp.drop_duplicates("title_norm", keep=False)[["title_norm", "openalex_id"]]
    w = w.merge(mp, on="title_norm", how="left").drop(columns=["title_norm"])
    cov = w["openalex_id"].notna().mean()
    w.to_parquet(os.path.join(OUT_DIR, f"work_{year}.parquet"), index=False)
    rows.append({"year": year, "n": len(w), "id_coverage": round(float(cov), 3),
                 "ambiguous_dropped": "title-level"})
    print(f"  {year}: n={len(w):,}  id coverage={cov:.1%}")
cov_df = pd.DataFrame(rows)
cov_df.to_csv(os.path.join(WORK_DIR, "openalex_id_attach_coverage.csv"), index=False)
print(f"\nMean id coverage: {cov_df['id_coverage'].mean():.1%}")
print("Enriched files -> ", OUT_DIR)

  1970: n=258,018  id coverage=89.8%
  1971: n=244,676  id coverage=89.6%
  1972: n=265,549  id coverage=89.4%
  1973: n=283,288  id coverage=90.0%
  1974: n=292,302  id coverage=89.9%
  1975: n=340,888  id coverage=90.4%
  1976: n=365,036  id coverage=90.5%
  1977: n=382,630  id coverage=90.5%
  1978: n=398,564  id coverage=90.5%
  1979: n=425,081  id coverage=90.4%
  1980: n=452,068  id coverage=90.2%
  1981: n=470,310  id coverage=90.4%
  1982: n=482,347  id coverage=89.8%
  1983: n=514,996  id coverage=89.8%
  1984: n=541,647  id coverage=89.8%
  1985: n=570,461  id coverage=90.3%
  1986: n=600,024  id coverage=90.3%
  1987: n=644,103  id coverage=90.1%
  1988: n=676,212  id coverage=90.1%
  1989: n=753,385  id coverage=90.1%
  1990: n=796,605  id coverage=90.2%
  1991: n=812,548  id coverage=89.7%
  1992: n=849,903  id coverage=89.9%
  1993: n=889,149  id coverage=90.5%
  1994: n=926,091  id coverage=90.6%
  1995: n=974,466  id coverage=90.2%
  1996: n=1,027,255  id coverage=88.2%

In [4]:
# ── Step 3: verify recovered ids join to sciscinet (paperid = W...) ──
SCI = os.path.join(SCI_DIR, "sciscinet_papers.parquet")
chk_year = 2000
w = pd.read_parquet(os.path.join(OUT_DIR, f"work_{chk_year}.parquet"), columns=["openalex_id"]).dropna()
ids = w["openalex_id"].astype(str).unique()[:20000]
idtype = pq.read_schema(SCI).field("paperid").type
vals = pa.array([str(x) for x in ids], type=idtype)
hit = ds.dataset(SCI).to_table(columns=["paperid"], filter=ds.field("paperid").isin(vals)).num_rows
print(f"{chk_year}: sampled {len(ids):,} recovered ids -> {hit:,} found in sciscinet ({hit/len(ids):.1%})")
print("If high, non-PPP papers sampled from work_by_year_with_id/ now carry sciscinet-joinable ids.")

2000: sampled 20,000 recovered ids -> 19,480 found in sciscinet (97.4%)
If high, non-PPP papers sampled from work_by_year_with_id/ now carry sciscinet-joinable ids.
